# PhishRoute v3 — Final decisive, cross-corpus evaluation

**Study status:** post-hoc protocol defined after v1/v2; not preregistered. The Notebook runs one comprehensive real-data study; it contains **no synthetic fallback**. Results must be interpreted before a Scientia Iranica submission.

## Required Kaggle Add Input (same as v2)

1. PhiUSIIL CSV or prior PhishRoute v1 ZIP (`PhishRoute_results.zip`).
2. LegitPhish V2 `url_features_extracted1.csv`.
3. URL-Phish V2 `URL-Phish_Dataset.csv` or `Dataset.csv` under an URL-Phish folder.

**Reproducibility:** Seeds 42, 52, 62. Four fixed representations, three source-calibrated FPR thresholds and four overlap policies. Strict-all-domain exclusion is the only primary independent transfer estimate. No live website scraping.


In [ ]:
from pathlib import Path
import subprocess, sys, importlib.util
if importlib.util.find_spec('tldextract') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','tldextract>=5,<6'])
print('Dependencies available. Input datasets must be attached with Kaggle Add Input.')


In [ ]:
%%writefile /kaggle/working/feature_engineering_v2.py
#!/usr/bin/env python3
"""PhishRoute v2: preregistered-style (NOT actually preregistered) artifact ablation and cross-corpus URL study.
Primary: UCI PhiUSIIL; external: Mendeley LegitPhish v2 and URL-Phish v2 (both REQUIRED).
Internal 1=phishing. PhiUSIIL/LegitPhish source label 0=phishing; URL-Phish label 1=phishing.
Only representation changes are studied; these do NOT imply URL semantic equivalence.
"""
from __future__ import annotations
import argparse, csv, hashlib, io, json, math, os, re, sys, time, warnings, zipfile
from pathlib import Path
from collections import Counter
from urllib.parse import urlsplit
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, f1_score, matthews_corrcoef, average_precision_score, roc_auc_score, brier_score_loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED=42
FP_COST=10
FN_COST=1
FPR_CAL_TARGET=.01
REPS=400
MIN_EXTERNAL_EACH_CLASS=50

# no unverified provider-specific numerical features: only shared URL lexical representation
LEX_NAMES=['url_len','host_len','path_len','query_len','fragment_len','domain_dots',
 'subdomain_segments','count_digits','frac_digits','frac_letters','count_hyphen',
 'count_dot','count_at','count_equal','count_question','count_ampersand','count_percent',
 'count_slash','count_under','count_tilde','count_colon','count_hash','count_semicolon',
 'is_https','host_is_ip','contains_punycode','contains_port','url_entropy',
 'host_entropy','url_token_count','host_hyphens']
ART_NAMES=['https_www_prefix','is_https','host_www_prefix','has_path','has_query','host_is_root','url_len']


def _hash(path):
 h=hashlib.sha256()
 with open(path,'rb') as fh:
  for chunk in iter(lambda:fh.read(2**20),b''):h.update(chunk)
 return h.hexdigest()


def _scan_files():
 roots=[Path('/kaggle/input'),Path('/kaggle/working'),Path('/mnt/data'),Path('.')]
 seen=set()
 for root in roots:
  if not root.exists():continue
  for p in root.rglob('*'):
   if not p.is_file() or p.suffix.lower() not in ('.csv','.zip'):continue
   if str(p) in seen:continue
   seen.add(str(p));yield p


def _lookup(kind,explicit):
 if explicit:
  p=Path(explicit)
  if not p.is_file():raise FileNotFoundError(f'Input missing: {p}')
  return p
 names={'primary':['phiusiil_phishing_url_dataset.csv','phiusiil_source.zip','phishroute_results.zip'],
        'external':['url_features_extracted1.csv','legitphish dataset.csv','legitphish.csv','legitphish.zip'],
        'external2':['url-phish_dataset.csv','url-phish.csv','urlphish.csv','urlphish.zip']}
 found=[]
 for p in _scan_files():
  n=p.name.lower()
  if n in names[kind]:found.append(p)
  elif kind=='external2' and n=='dataset.csv' and any(a in str(p).lower() for a in ['url-phish','urlphish','65z9twcx3r']):found.append(p)
 if not found:
  need=({'primary':'PhiUSIIL_Phishing_URL_Dataset.csv (UCI #967)',
         'external':'url_features_extracted1.csv (LegitPhish V2 DOI 10.17632/hx4m73v2sf.2)',
         'external2':'Dataset.csv from URL-Phish V2 DOI 10.17632/65z9twcx3r.2; rename it URL-Phish_Dataset.csv if needed'}[kind])
  raise FileNotFoundError(f'{kind} dataset not found. Add {need} to Kaggle > Add Input. NO synthetic fallback.')
 found.sort(key=lambda p:(len(str(p)),str(p)))
 print('Selected',kind,':',found[0]);return found[0]


def read_dataset(path,kind):
 p=Path(path)
 if p.suffix.lower()=='.csv':return pd.read_csv(p,low_memory=False),str(p),_hash(p)
 if p.suffix.lower()=='.zip':
  with zipfile.ZipFile(p) as z:
   want='phiusiil' if kind=='primary' else ('url_features_extracted1' if kind=='external' else 'dataset.csv')
   matches=[n for n in z.namelist() if n.lower().endswith('.csv') and want in n.lower()]
   if kind=='external' and not matches:matches=[n for n in z.namelist() if n.lower().endswith('.csv') and 'legitphish' in n.lower()]
   if not matches:raise ValueError(f'Cannot find the {kind} CSV inside ZIP {p}.')
   with z.open(matches[0]) as fh:
    data=fh.read()
   return pd.read_csv(io.BytesIO(data),low_memory=False),f'{p}::{matches[0]}',hashlib.sha256(data).hexdigest()
 raise ValueError('Expected .csv or .zip')


def get_columns(df,kind):
 colmap={c.lower().replace(' ','').replace('_',''):c for c in df.columns}
 if 'url' not in colmap:raise ValueError(f'{kind}: URL column missing; got {list(df.columns)}')
 key=['label'] if kind in ('primary','external2') else ['classlabel','label']
 ycol=next((colmap[k] for k in key if k in colmap),None)
 if not ycol:raise ValueError(f'{kind}: label column missing; got {list(df.columns)}')
 return colmap['url'],ycol


def clean_urls(raw,kind):
 ucol,lcol=get_columns(raw,kind)
 n0=len(raw)
 x=raw[[ucol,lcol]].copy().rename(columns={ucol:'URL',lcol:'raw_label'})
 x['raw_label']=pd.to_numeric(x.raw_label,errors='coerce')
 x.URL=x.URL.astype('string').str.strip()
 x=x[x.raw_label.isin([0.,1.])&x.URL.notna()&x.URL.str.len().gt(0)].copy()
 n_invalid=n0-len(x)
 conflicts=x.groupby('URL',sort=False).raw_label.nunique()
 conflict_urls=set(conflicts.index[conflicts.gt(1)])
 n_conflict=int(x.URL.isin(conflict_urls).sum())
 x=x[~x.URL.isin(conflict_urls)].drop_duplicates('URL',keep='first').reset_index(drop=True)
 x['y']=(x.raw_label.eq(1) if kind=='external2' else x.raw_label.eq(0)).astype('int8')
 return x,{'raw_rows':n0,'invalid_rows':n_invalid,'conflict_rows':n_conflict,
           'deduplicated_rows':len(x),'duplicate_or_conflict_removed':n0-n_invalid-n_conflict-len(x),
           'positive_convention':('1=phishing; source 1=phishing' if kind=='external2' else '1=phishing; source 0=phishing; source 1=legitimate')}


def parse_url(raw):
 s=str(raw).strip()
 if not re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://',s): s='http://'+s
 try:
  p=urlsplit(s)
  host=(p.hostname or '').lower().strip('.')
 except (ValueError,OverflowError):
  p=urlsplit('http://invalid.local/');host='invalid.local'
 return p,host


def reg_domain(host,extractor):
 r=extractor(host)
 return (r.domain+'.'+r.suffix).lower() if r.suffix else host


def entropy(s):
 if not s:return 0.
 c=Counter(s);n=len(s)
 return -sum((v/n)*math.log2(v/n) for v in c.values())


def feature_tuple(u):
 p,host=parse_url(u)
 lower=str(u).lower();base=lower.split('://',1)[-1];n=max(1,len(base))
 try:
  import ipaddress
  ipaddress.ip_address(host.strip('[]'));ip=1
 except ValueError:ip=0
 try:port=int(p.port is not None)
 except ValueError:port=0
 out=[len(str(u)),len(host),len(p.path),len(p.query),len(p.fragment),host.count('.'),
 max(len(host.split('.'))-2,0),sum(c.isdigit() for c in base),sum(c.isdigit() for c in base)/n,
 sum(c.isalpha() for c in base)/n,base.count('-'),base.count('.'),base.count('@'),
 base.count('='),base.count('?'),base.count('&'),base.count('%'),base.count('/'),
 base.count('_'),base.count('~'),base.count(':'),base.count('#'),base.count(';'),
 int(p.scheme.lower()=='https'),ip,int('xn--' in host),port,entropy(base),entropy(host),
 len([a for a in re.split(r'[^a-z0-9]+',base) if a]),host.count('-')]
 www=int(host.startswith('www.'))
 art=[int(str(u).lower().startswith('https://www.')),int(p.scheme=='https'),www,
 int(bool(p.path and p.path!='/')),int(bool(p.query)),int(not bool(p.path and p.path!='/') and not bool(p.query)),len(str(u))]
 canonical_host=re.sub(r'^www\.', '', host)
 canonical='http://'+canonical_host+(p.path or '')+(('?'+p.query) if p.query else '')+(('#'+p.fragment) if p.fragment else '')
 host_only='http://'+canonical_host
 return out,feature_tuple_basic(canonical),feature_tuple_basic(host_only),art,host


def feature_tuple_basic(u):
 p,host=parse_url(u)
 lower=str(u).lower();base=lower.split('://',1)[-1];n=max(1,len(base))
 try:
  import ipaddress
  ipaddress.ip_address(host.strip('[]'));ip=1
 except ValueError:ip=0
 try:port=int(p.port is not None)
 except ValueError:port=0
 return [len(str(u)),len(host),len(p.path),len(p.query),len(p.fragment),host.count('.'),
 max(len(host.split('.'))-2,0),sum(c.isdigit() for c in base),sum(c.isdigit() for c in base)/n,
 sum(c.isalpha() for c in base)/n,base.count('-'),base.count('.'),base.count('@'),
 base.count('='),base.count('?'),base.count('&'),base.count('%'),base.count('/'),
 base.count('_'),base.count('~'),base.count(':'),base.count('#'),base.count(';'),
 int(p.scheme.lower()=='https'),ip,int('xn--' in host),port,entropy(base),entropy(host),
 len([a for a in re.split(r'[^a-z0-9]+',base) if a]),host.count('-')]


def make_features(frame,extractor):
 n=len(frame)
 print('Extracting URL representations for',n,'rows',flush=True)
 raw=np.empty((n,len(LEX_NAMES)),dtype=np.float32)
 canon=np.empty_like(raw);host=np.empty_like(raw)
 artifacts=np.empty((n,len(ART_NAMES)),dtype=np.float32)
 groups=np.empty(n,dtype=object)
 for i,u in enumerate(frame.URL.tolist()):
  a,b,c,d,h=feature_tuple(u)
  raw[i]=a;canon[i]=b;host[i]=c;artifacts[i]=d
  groups[i]=reg_domain(h,extractor) or 'MISSING_'+str(i)
  if (i+1)%100000==0:print('...processed',i+1,flush=True)
 return {'raw':raw,'presentation_ablated':canon,'hostname_only':host,'artifact_only':artifacts},groups


def split_primary(y,groups):
 indices=np.arange(len(y))
 train_cal,test=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(indices,y,groups))
 t,c=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED+1).split(train_cal,y[train_cal],groups[train_cal]))
 train=train_cal[t];cal=train_cal[c]
 for a,b in [('train','test'),('train','cal'),('cal','test')]:
  ids={'train':train,'cal':cal,'test':test}
  if set(groups[ids[a]])&set(groups[ids[b]]):raise RuntimeError('Domain leakage detected')
 if any(len(np.unique(y[z]))!=2 for z in [train,cal,test]):raise RuntimeError('Invalid split with one class')
 return dict(train=train,cal=cal,test=test)


def threshold_at_cal_fpr(y,p):
 vals=np.asarray(p)[np.asarray(y)==0]
 if len(vals)<100:raise ValueError('Too few calibration negatives')
 return float(np.nextafter(np.quantile(vals,1-FPR_CAL_TARGET,method='higher'),np.inf))


def evaluate(y,pred,threshold,model,corpus,condition):
 y=np.asarray(y,dtype='int8'); p=np.asarray(pred,dtype=float)
 z=(p>=threshold).astype('int8');tn,fp,fn,tp=confusion_matrix(y,z,labels=[0,1]).ravel()
 n=len(y)
 out=dict(model=model,corpus=corpus,representation=condition,threshold=float(threshold),n=n,
   positives=int(np.sum(y,dtype=np.int64)),prevalence=float(y.mean()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),
   accuracy=float((tp+tn)/n),FPR=float(fp/max(fp+tn,1)),
   precision=float(tp/max(tp+fp,1)),recall=float(tp/max(tp+fn,1)),
   F1=float(f1_score(y,z,zero_division=0)),MCC=float(matthews_corrcoef(y,z)),
   ROC_AUC=float(roc_auc_score(y,p)) if len(set(y))==2 else None,
   PR_AUC=float(average_precision_score(y,p)) if len(set(y))==2 else None,
   Brier=float(brier_score_loss(y,p)),
   cost_per_1000=float(1000*(FP_COST*fp+FN_COST*fn)/n))
 return out,z


def format_audit(art,y,corpus):
 rows=[]
 for name in ['https_www_prefix','is_https','host_www_prefix','has_path','has_query','host_is_root']:
  j=ART_NAMES.index(name)
  for cl,kind in [(0,'legitimate'),(1,'phishing')]:
   a=art[np.asarray(y)==cl,j]
   rows.append(dict(corpus=corpus,class_name=kind,indicator=name,n=len(a),rate=float(a.mean())))
 return rows


def cluster_bootstrap(y,groups,za,zb,reps=REPS):
 y=np.asarray(y);za=np.asarray(za);zb=np.asarray(zb)
 ids,guniq=pd.factorize(groups,sort=False);g=len(guniq)
 def ct(z):
  tp=np.bincount(ids,weights=((y==1)&(z==1)).astype(int),minlength=g)
  tn=np.bincount(ids,weights=((y==0)&(z==0)).astype(int),minlength=g)
  fp=np.bincount(ids,weights=((y==0)&(z==1)).astype(int),minlength=g)
  fn=np.bincount(ids,weights=((y==1)&(z==0)).astype(int),minlength=g)
  return np.column_stack([tp,tn,fp,fn])
 aa=ct(za);bb=ct(zb);rng=np.random.default_rng(SEED+101)
 arr=np.empty((reps,3),dtype=float)
 for k in range(reps):
  sampled=rng.integers(0,g,size=g)
  a=aa[sampled].sum(axis=0);b=bb[sampled].sum(axis=0)
  def f1(x):return 2*x[0]/max(2*x[0]+x[2]+x[3],1)
  def fpr(x):return x[2]/max(x[1]+x[2],1)
  def loss(x):return 1000*(10*x[2]+x[3])/max(x.sum(),1)
  arr[k]=[f1(a)-f1(b),fpr(a)-fpr(b),loss(a)-loss(b)]
 result={}
 for j,nm in enumerate(['F1_delta_ablated_minus_raw','FPR_delta_ablated_minus_raw','cost_delta_ablated_minus_raw']):
  result[nm]=dict(median=float(np.median(arr[:,j])),CI95=[float(v) for v in np.quantile(arr[:,j],[.025,.975])])
 return result


def plots(perf,audit,out):
 plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,'axes.spines.right':False})
 bas=perf[perf.model.isin(['raw','presentation_ablated','hostname_only','artifact_only'])]
 corps=list(dict.fromkeys(bas.corpus.tolist()))
 if not len(corps):return
 for metric,file,ylabel in [('FPR','Fig1_FPR','False positive rate'),('recall','Fig2_Recall','Phishing recall'),('F1','Fig3_F1','F1 score')]:
  names=['raw','presentation_ablated','hostname_only','artifact_only'];styles=['o','s','^','D'];fig,ax=plt.subplots(figsize=(7.1,4.2))
  x=np.arange(len(corps));w=.18
  for j,(name,m) in enumerate(zip(names,styles)):
   vals=[]
   for c in corps:
    row=bas[(bas.corpus==c)&(bas.model==name)]
    vals.append(float(row.iloc[0][metric]) if len(row) else np.nan)
   ax.bar(x+(j-1.5)*w,vals,w,label=name.replace('_',' ').title(),facecolor=['#222','#777','#aaa','#ddd'][j],edgecolor='black',linewidth=.6)
  ax.set_xticks(x);ax.set_xticklabels(corps,rotation=0)
  ax.set_ylabel(ylabel);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
  ax.legend(loc='upper center',bbox_to_anchor=(.5,1.17),ncol=2,frameon=False,fontsize=8)
  fig.tight_layout();fig.savefig(out/(file+'.png'),dpi=300,bbox_inches='tight');fig.savefig(out/(file+'.svg'),bbox_inches='tight');plt.close(fig)
  
 fig,ax=plt.subplots(figsize=(8.2,4.3))
 ind=['https_www_prefix','has_path','has_query'];x=np.arange(len(ind))
 comparisons=[(corp,cls) for corp in corps for cls in ['legitimate','phishing']]
 w=.78/max(len(comparisons),1)
 gray=np.linspace(.12,.85,len(comparisons))
 for j,(corpus,cl) in enumerate(comparisons):
  vals=[]
  for name in ind:
   r=audit[(audit.corpus==corpus)&(audit.class_name==cl)&(audit.indicator==name)]
   vals.append(float(r.iloc[0].rate) if len(r) else np.nan)
  ax.bar(x+(j-(len(comparisons)-1)/2)*w,vals,width=w,
   label=f'{corpus} / {cl}',color=str(gray[j]),edgecolor='black',linewidth=.5)
 ax.set_xticks(x);ax.set_xticklabels(['HTTPS + WWW','Non-root path','Query string']);ax.set(ylabel='Frequency within class',ylim=(0,1.05))
 ax.legend(frameon=False,fontsize=7,ncol=3,loc='upper center',bbox_to_anchor=(.5,1.24));ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
 fig.tight_layout();fig.savefig(out/'Fig4_CollectionArtifacts.png',dpi=300,bbox_inches='tight');fig.savefig(out/'Fig4_CollectionArtifacts.svg',bbox_inches='tight');plt.close(fig)


def main():
 ap=argparse.ArgumentParser()
 ap.add_argument('--primary',help='Original PhiUSIIL CSV or PhishRoute_results.zip')
 ap.add_argument('--external',help='Original LegitPhish V2 CSV or ZIP (required)')
 ap.add_argument('--external2',help='Original URL-Phish V2 Dataset.csv or ZIP (required; label coding differs)')
 ap.add_argument('--out',default='/kaggle/working/PhishRoute_v2_results' if Path('/kaggle').exists() else './PhishRoute_v2_results')
 ap.add_argument('--max-rows',type=int,default=0,help='Smoke testing only; NOT publication evidence')
 args=ap.parse_args()
 t=time.perf_counter();out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
 is_smoke=args.max_rows>0
 specs=[('PhiUSIIL','primary',args.primary),('LegitPhish','external',args.external),('URL-Phish','external2',args.external2)]
 data={};source_metadata={}
 # Enforce all sources before modeling, preventing another iterative run.
 for name,kind,path in specs:
  src=_lookup(kind,path);df,srcname,checksum=read_dataset(src,kind)
  cleaned,clean_info=clean_urls(df,kind)
  if is_smoke:
   cleaned=pd.concat([cleaned[cleaned.y==0].head(args.max_rows//2),
                      cleaned[cleaned.y==1].head(args.max_rows//2)],ignore_index=True)
  data[name]=cleaned
  source_metadata[name]=dict(path=srcname,sha256=checksum,cleaning=clean_info)
 try:
  import tldextract
 except ImportError as ex:
  raise RuntimeError('Install tldextract (pip install tldextract>=5). Scientific splits require the Public Suffix List; hostname-only fallback is forbidden.') from ex
 extractor=tldextract.TLDExtract(suffix_list_urls=())
 features={};groups={};labels={}
 for name in data:
  features[name],groups[name]=make_features(data[name],extractor)
  labels[name]=data[name].y.to_numpy('int8')
 py=labels['PhiUSIIL'];pg=groups['PhiUSIIL'];split=split_primary(py,pg)
 primary_domains=set(pg)
 overlap_report={}
 # Exclude ALL primary domains and exact URLs from each external corpus, not just train domains.
 for name in ['LegitPhish','URL-Phish']:
  g=groups[name]
  mask=np.asarray([d not in primary_domains for d in g],dtype=bool)
  before=len(g);removed=int(np.sum(~mask))
  exact=len(set(data['PhiUSIIL'].URL.astype(str)) & set(data[name].URL.astype(str)))
  data[name]=data[name].iloc[np.flatnonzero(mask)].reset_index(drop=True)
  features[name]={k:v[mask] for k,v in features[name].items()}
  groups[name]=g[mask];labels[name]=labels[name][mask]
  cc=np.bincount(labels[name],minlength=2)
  per_class_domains={str(k):len(set(groups[name][labels[name]==k])) for k in (0,1)}
  overlap_report[name]={'initial_rows':before,'excluded_primary_overlap_rows':removed,
                        'exact_primary_url_overlaps_before_filter':exact,
                        'class_counts_after_filter':cc.tolist(),
                        'distinct_registered_domains_by_class':per_class_domains}
  if not is_smoke and min(cc)<MIN_EXTERNAL_EACH_CLASS:
   raise RuntimeError(f'{name}: independent sample has too few cases after domain overlap removal: {cc}. The corpus is NOT valid for two-class external evaluation.')
  if min(cc)<5:raise RuntimeError(f'{name}: class count too low for smoke test: {cc}')
 overlap_report['between_external_sources_distinct_domains']=len(set(groups['LegitPhish'])&set(groups['URL-Phish']))
 split_summary={name:dict(rows=int(len(ix)),domains=int(len(set(pg[ix]))),phishing_rate=float(py[ix].mean())) for name,ix in split.items()}
 for name in ['LegitPhish','URL-Phish']:
  split_summary[name]=dict(rows=int(len(labels[name])),domains=len(set(groups[name])),phishing_rate=float(labels[name].mean()))
 print('Group split:',json.dumps(split_summary,indent=2),flush=True)
 audit_rows=[]
 for name in ['PhiUSIIL','LegitPhish','URL-Phish']:
  audit_rows+=format_audit(features[name]['artifact_only'],labels[name],name)
 audit=pd.DataFrame(audit_rows);audit.to_csv(out/'Table1_ArtifactFrequencies.csv',index=False)
 predictions={};metrics=[];thresholds={};model_card={}
 for rep in ['raw','presentation_ablated','hostname_only','artifact_only']:
  print('Training representation:',rep,flush=True)
  if rep=='artifact_only':
   from sklearn.pipeline import make_pipeline
   from sklearn.preprocessing import StandardScaler
   model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED))
  else:
   model=HistGradientBoostingClassifier(max_leaf_nodes=25,max_depth=5,
    min_samples_leaf=3 if is_smoke else 30,max_iter=20 if is_smoke else 130,
    learning_rate=.07,l2_regularization=1.,early_stopping=False,random_state=SEED)
  model.fit(features['PhiUSIIL'][rep][split['train']],py[split['train']])
  p_cal=model.predict_proba(features['PhiUSIIL'][rep][split['cal']])[:,1]
  if is_smoke:
   vals=p_cal[py[split['cal']]==0]
   threshold=float(np.quantile(vals,.99))
  else:threshold=threshold_at_cal_fpr(py[split['cal']],p_cal)
  thresholds[rep]=threshold
  for name in ['PhiUSIIL','LegitPhish','URL-Phish']:
   ix=split['test'] if name=='PhiUSIIL' else np.arange(len(labels[name]))
   y=labels[name][ix];g=groups[name][ix]
   probs=model.predict_proba(features[name][rep][ix])[:,1]
   row,z=evaluate(y,probs,threshold,rep,name,rep)
   metrics.append(row);predictions[(name,rep)]=(y,g,z,probs)
  model_card[rep]={'n_features':int(features['PhiUSIIL'][rep].shape[1]),
      'train_rows':int(len(split['train'])),'calibration_FPR_target':.01}
 # The fixed post-hoc rule is a DIAGNOSTIC comparator, never a deployed phishing detector.
 for name in ['PhiUSIIL','LegitPhish','URL-Phish']:
  ix=split['test'] if name=='PhiUSIIL' else np.arange(len(labels[name]))
  art=features[name]['artifact_only'];y=labels[name][ix]
  score=1-art[ix,ART_NAMES.index('https_www_prefix')]
  row,z=evaluate(y,score,.5,'fixed_https_www_rule',name,'artifact_diagnostic')
  metrics.append(row)
 perf=pd.DataFrame(metrics);perf.to_csv(out/'Table2_PrimaryAndExternalPerformance.csv',index=False)
 boots={}
 if not is_smoke:
  for name in ['PhiUSIIL','LegitPhish','URL-Phish']:
   y,g,z_raw,_=predictions[(name,'raw')]
   _,_,z_abl,_=predictions[(name,'presentation_ablated')]
   unique_class_groups={str(k):len(set(g[y==k])) for k in (0,1)}
   if min(unique_class_groups.values())<10:
    boots[name]={'not_computed':'Fewer than 10 unique domain groups in at least one class',
                 'domains_by_class':unique_class_groups}
   else:boots[name]=cluster_bootstrap(y,g,z_abl,z_raw)
 (out/'Table3_PairedClusterBootstrap.json').write_text(json.dumps(boots,indent=2))
 plots(perf,audit,out)
 from sklearn import __version__ as skversion
 manifest={'study':'PhishRoute_v2','status':'SMOKE_ONLY_NOT_SCIENTIFIC' if is_smoke else 'EXECUTED_THREE_SOURCE_STUDY',
  'claims_restricted':True,'datasets':source_metadata,'domain_overlap':overlap_report,'split':split_summary,
  'internal_label_mapping':'1=phishing; PhiUSIIL and LegitPhish original 0=phishing; URL-Phish original 1=phishing',
  'model_definitions':model_card,'thresholds':thresholds,
  'seed':SEED,'bootstrap_reps':REPS if not is_smoke else 0,
  'calibration':'Calibrated at 1% empirical FPR on primary calibration negatives; NOT guaranteed external FPR',
  'environment':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,'sklearn':skversion},
  'limitations':['Cross-source dataset provenance and calendar time are confounded',
    'Presentation ablation is a representation test, NOT a causal intervention on website behavior',
    'Corpora use different collection and labeling processes',
    'No reliable phishing timestamps in primary for temporal holdout',
    'Format-only diagnostic discovered post hoc in v1; not preregistered',
    'No claims about network latency or live phishing detection',
    'Two external benchmarks may share some registrable domains with each other',
    'Cluster bootstrap cannot repair systematic corpus biases'],
  'elapsed_seconds':float(time.perf_counter()-t)}
 (out/'MANIFEST.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf8')
 primary_leg=audit[(audit.corpus=='PhiUSIIL')&(audit.class_name=='legitimate')&(audit.indicator=='https_www_prefix')].rate.iloc[0]
 primary_ph=audit[(audit.corpus=='PhiUSIIL')&(audit.class_name=='phishing')&(audit.indicator=='https_www_prefix')].rate.iloc[0]
 gate={'real_external_corpora':2,'all_external_primary_registered_domains_excluded':True,
  'collection_format_gap_in_primary':float(primary_leg-primary_ph),
  'number_distinct_external_groups':{name:int(len(set(groups[name]))) for name in ['LegitPhish','URL-Phish']},
  'submission_ready':False,'note':'Interpret effect sizes and literature novelty manually; never claim a successful new algorithm automatically.'}
 (out/'EVIDENCE_GATE.json').write_text(json.dumps(gate,indent=2),encoding='utf8')
 (out/'RUN_STATUS.txt').write_text('PhishRoute v2 execution finished. Status: '+manifest['status']+'\n'
  'Three separate real datasets required. Cross-corpus errors do not establish deployment FPR.\n'
  'A source-specific artifact is not necessarily a causal shortcut.\n',encoding='utf8')
 import shutil
 shutil.make_archive(str(out),'zip',root_dir=out)
 print('\n=== RESULTS ===\n',perf[['corpus','model','FPR','recall','F1','MCC','cost_per_1000']].to_string(index=False))
 print('ZIP:',str(out)+'.zip');return 0

if __name__=='__main__':
 try:sys.exit(main())
 except Exception:
  import traceback;traceback.print_exc();sys.exit(1)


In [ ]:
%%writefile /kaggle/working/run_decisive_study.py
#!/usr/bin/env python3
"""PhishRoute v3 final, one-pass evidence study.

Source-trained domain-disjoint cross-corpus tests, overlap policy sensitivity,
repeated group-split stability, calibrated operating-point sensitivity.
No claim of prospective registration, causal effect, production FPR, or novelty.
"""
from __future__ import annotations
import argparse, hashlib, json, math, os, sys, time, warnings, zipfile, shutil
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import feature_engineering_v2 as legacy

CORPORA=('PhiUSIIL','LegitPhish','URL-Phish')
ORIGINS=('PhiUSIIL','URL-Phish') # LegitPhish lacks diverse disjoint legitimate domains
REPS=('raw','presentation_ablated','hostname_only','artifact_only')
CAL_TARGETS=(.005,.01,.02)
SEEDS=(42,52,62)
POLICIES=('strict_all','training_cal_domains','exact_urls','unfiltered')
MIN_CLASS_FOR_INFERENCE=100
MIN_GROUPS_FOR_INFERENCE=20
MIN_CLASS_RETENTION=0.10  # descriptive safeguard after severe class depletion seen in v2


def split_groups(y,g,seed,min_cal_neg=100):
    idx=np.arange(len(y))
    p, test=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=seed).split(idx,y,g))
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=seed+1).split(p,y[p],g[p]))
    tr,ca=p[a],p[b]
    for x,z in ((tr,ca),(tr,test),(ca,test)):
        assert not set(g[x]).intersection(g[z]),'SPLIT_DOMAIN_OVERLAP'
    assert all(len(np.unique(y[v]))==2 for v in (tr,ca,test)),'SPLIT_SINGLE_CLASS'
    assert sum(y[ca]==0)>=min_cal_neg,'TOO_FEW_CALIBRATION_NEGATIVES'
    return {'train':tr,'cal':ca,'test':test}


def cal_threshold(y,prob,target,min_cal_neg=100):
    neg=np.asarray(prob)[np.asarray(y)==0]
    if len(neg)<min_cal_neg:raise ValueError('Insufficient calibration negatives')
    # Choose the smallest score strictly above the (1-target) quantile.
    # Guarantees empirical FPR at or below target on this calibration subset;
    # ties can make the realized calibration FPR lower than target.
    v=float(np.quantile(neg,1-target,method='higher'))
    return float(np.nextafter(v, np.inf))


def overlap_mask(origin,target,policy,train_cal_indices, data,groups):
    if origin==target:raise ValueError('Self is not an external test')
    if policy=='strict_all':
        blocked=set(groups[origin]);return np.fromiter((d not in blocked for d in groups[target]),bool,count=len(groups[target]))
    if policy=='training_cal_domains':
        blocked=set(groups[origin][train_cal_indices]);return np.fromiter((d not in blocked for d in groups[target]),bool,count=len(groups[target]))
    if policy=='exact_urls':
        blocked=set(data[origin].URL.tolist());return (~data[target].URL.isin(blocked)).to_numpy(bool)
    if policy=='unfiltered':return np.ones(len(groups[target]),dtype=bool)
    raise ValueError('Unknown overlap policy')


def label_counts(y,g):
    y=np.asarray(y);g=np.asarray(g)
    return {'n':int(len(y)), 'n_legit':int((y==0).sum()),'n_phishing':int((y==1).sum()),
            'distinct_domains':int(len(set(g))),
            'domains_legit':int(len(set(g[y==0]))),'domains_phishing':int(len(set(g[y==1]))),
            'phishing_prevalence':float(y.mean()) if len(y) else None}


def is_interpretable(a):
    return a['n_legit']>=MIN_CLASS_FOR_INFERENCE and a['n_phishing']>=MIN_CLASS_FOR_INFERENCE and min(a['domains_legit'],a['domains_phishing'])>=MIN_GROUPS_FOR_INFERENCE

def class_retention(c,original_y):
    n0=max(int(np.sum(original_y==0)),1)
    n1=max(int(np.sum(original_y==1)),1)
    return c['n_legit']/n0,c['n_phishing']/n1

def strict_inference_quality(c,original_y):
    retention=class_retention(c,original_y)
    return is_interpretable(c) and min(retention)>=MIN_CLASS_RETENTION


def safe_evaluate(y,prob,threshold,**meta):
    if not len(y):return {**meta,'n':0,'evaluable':False}
    if len(set(y))<2:
        # Both-class metrics are not meaningful, but counts must remain transparent.
        return {**meta,'n':int(len(y)),'n_legit':int((y==0).sum()),'n_phishing':int((y==1).sum()),'evaluable':False}
    row,z=legacy.evaluate(y,prob,threshold,meta.get('representation',''),meta.get('corpus',''),meta.get('representation',''))
    return {**row, **meta,'evaluable':True},z


def compare_conflicts(data):
    out=[]
    for i,left in enumerate(CORPORA):
        d1=data[left][['URL','y']]
        for right in CORPORA[i+1:]:
            d2=data[right][['URL','y']]
            joined=d1.merge(d2,on='URL',how='inner',suffixes=('_a','_b'))
            out.append({'first_corpus':left,'second_corpus':right,
                        'exact_url_overlap':int(len(joined)),
                        'contradictory_labels':int((joined.y_a!=joined.y_b).sum())})
    return out


def summary_rows(rows):
    df=pd.DataFrame(rows)
    if df.empty:return pd.DataFrame()
    good=df[(df.get('evaluable',False)==True)&(df['policy']=='strict_all')&(df['target_fpr']==.01)&(df['origin']!=df['corpus'])].copy()
    if good.empty:return pd.DataFrame()
    return good.groupby(['origin','corpus','representation'],sort=True).agg(
        n_median=('n','median'),fpr_median=('FPR','median'),fpr_min=('FPR','min'),fpr_max=('FPR','max'),
        recall_median=('recall','median'),recall_min=('recall','min'),recall_max=('recall','max'),
        f1_median=('F1','median'),mcc_median=('MCC','median'),
        cost_median=('cost_per_1000','median'),class_coverage_ok=('independent_class_diversity', 'all'),
        n_seeds=('seed','nunique')).reset_index()


def make_plots(df,summary,counts,out):
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.spines.top':False,'axes.spines.right':False})
    if not summary.empty:
        focus=summary[summary.representation.isin(['raw','presentation_ablated'])].copy()
        combos=sorted(set(zip(focus.origin,focus.corpus)))
        fig,axs=plt.subplots(2,1,figsize=(8.2,7.3),sharex=True)
        for ax,metric,lab in [(axs[0],'fpr_median','External FPR'),(axs[1],'recall_median','Phishing recall')]:
            x=np.arange(len(combos));w=.33
            for j,rep in enumerate(['raw','presentation_ablated']):
                vals=[]
                for source,dst in combos:
                    part=focus[(focus.origin==source)&(focus.corpus==dst)&(focus.representation==rep)]
                    vals.append(float(part.iloc[0][metric]) if len(part) else np.nan)
                ax.bar(x+(j-.5)*w,vals,w,color=['#252525','#9d9d9d'][j],edgecolor='black',linewidth=.5,label=rep.replace('_',' '))
            ax.set_ylim(0,1.05);ax.set_ylabel(lab);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
            ax.legend(frameon=False,loc='upper right')
        axs[1].set_xticks(range(len(combos)));axs[1].set_xticklabels([f'{a} to {b}' for a,b in combos],rotation=12)
        fig.tight_layout();fig.savefig(out/'Fig1_ReverseTransfer_FPR_Recall.png',dpi=300,bbox_inches='tight');fig.savefig(out/'Fig1_ReverseTransfer_FPR_Recall.svg',bbox_inches='tight');plt.close(fig)

    raw=df[(df.get('evaluable',False)==True)&(df.get('seed',pd.Series(dtype=float))==42)&
        (df.get('origin',pd.Series(dtype=str))=='PhiUSIIL')&(df.get('corpus',pd.Series(dtype=str))=='LegitPhish')&
        (df.get('representation',pd.Series(dtype=str))=='raw')&(df.get('target_fpr',pd.Series(dtype=float))==.01)]
    if len(raw):
        fig,axs=plt.subplots(1,2,figsize=(9,3.8));pols=list(POLICIES)
        for ax,m in [(axs[0],'FPR'),(axs[1],'recall')]:
            vals=[float(raw[raw.policy==p].iloc[0][m]) if any(raw.policy==p) else np.nan for p in pols]
            ax.bar(range(len(pols)),vals,color=['#222','#666','#999','#ccc'],edgecolor='black',linewidth=.6)
            ax.set_xticks(range(len(pols)));ax.set_xticklabels(['Strict all','Train+cal','Exact URL','Unfiltered'],rotation=25,ha='right')
            ax.set_ylabel(m);ax.set_ylim(0,1.05);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
        fig.tight_layout();fig.savefig(out/'Fig2_OverlapPolicy_Sensitivity.png',dpi=300,bbox_inches='tight');fig.savefig(out/'Fig2_OverlapPolicy_Sensitivity.svg',bbox_inches='tight');plt.close(fig)


def generate_brief(summary,primary_rows,meta,overlaps,out):
    lines=['# PhishRoute v3 — automatic evidence report (not a final manuscript)', '',
           'This report is generated from real execution; the publication and novelty verdict requires manual review.',
           '', '## Cross-source strict disjoint transfer at calibration FPR target 1%']
    if summary.empty:lines.append('No evaluable transfer tests. The research claim is unsupported.')
    else:
        lines.append('| Train -> Test | Model | Median external FPR | Median recall | FPR range across 3 splits |')
        lines.append('|---|---|---:|---:|---:|')
        for _,r in summary.iterrows():
            lines.append(f"| {r['origin']} -> {r['corpus']} | {r['representation']} | {r['fpr_median']:.3f} | {r['recall_median']:.3f} | {r['fpr_min']:.3f}–{r['fpr_max']:.3f} |")
    lines += ['', '## Dataset validity cautions',
              '- Exact-URL overlap and contradictory labels across public sources are counted separately.',
              '- Strict-all-domain filtering is the **only** primary independent external outcome.',
              '- Training/calibration-domain, exact URL, and unfiltered policies are overlap diagnostics, NOT independent confirmation.',
              '- High prevalence differences and source ascertainment invalidate prevalence-naive cross-source F1 interpretation.',
              '- Three seeds are robustness checks on related samples; they are not independent external experiments.',
              '- Thresholds are calibrated ONLY in the training source at 0.5%, 1%, and 2% empirical FPR targets.',
              '- A low FPR in the calibration partition does NOT ensure a low FPR in unseen sources.',
              '- LegitPhish benign coverage after strict filtering may be tiny, producing fragile FPR estimates.',
              '- Representation masking changes feature extraction only; it does not alter website reality.',
              '- No experiment here establishes a causally effective correction or a novel detector.',
              '- Sources are collected from different ecosystems and periods; there is no temporal benchmark.',
              '', '## Data provenance',
              'PhiUSIIL: UCI 967; LegitPhish v2: 10.17632/hx4m73v2sf.2; URL-Phish v2: 10.17632/65z9twcx3r.2.',
              'Full SHA256 fingerprints, dataset sizes, model parameters and exact splits are in MANIFEST.json.',
              '', '## Publication decision',
              'NOT submission ready automatically. Compare novelty against ACIT 2025 dataset-quality/generalization study and subsequent external works.']
    (out/'AUTOMATIC_EVIDENCE_REPORT_EN.md').write_text('\n'.join(lines)+'\n',encoding='utf8')


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--primary');ap.add_argument('--external');ap.add_argument('--external2')
    ap.add_argument('--out',default='/kaggle/working/PhishRoute_v3_final' if Path('/kaggle').exists() else './PhishRoute_v3_final')
    ap.add_argument('--smoke-test',action='store_true',help='Synthetic test only; NO SCIENTIFIC CONCLUSIONS')
    args=ap.parse_args()
    st=time.perf_counter();out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    filenames=[('PhiUSIIL','primary',args.primary),('LegitPhish','external',args.external),('URL-Phish','external2',args.external2)]
    data,metadata={},{}
    for name,kind,explicit in filenames:
        src=legacy._lookup(kind,explicit)
        table,location,sha=legacy.read_dataset(src,kind)
        clean,stats=legacy.clean_urls(table,kind)
        if args.smoke_test:
            # A smoke test may be run only on explicitly synthetic datasets supplied in tests.
            clean=pd.concat([clean.loc[clean.y==0].head(120),clean.loc[clean.y==1].head(120)],ignore_index=True)
        data[name]=clean;metadata[name]={'input':location,'sha256':sha,'cleaning':stats}
        print('Loaded',name,'rows',len(clean),flush=True)
    try:
        import tldextract
    except ImportError as e:
        raise RuntimeError('Missing tldextract: Kaggle run must install tldextract>=5,<6') from e
    extractor=tldextract.TLDExtract(suffix_list_urls=())
    feat,groups,labels={},{},{}
    for name in CORPORA:
        feat[name],groups[name]=legacy.make_features(data[name],extractor)
        labels[name]=data[name].y.to_numpy('int8')
    assert all(len(set(labels[name]))==2 for name in CORPORA),'CORPUS_WITH_SINGLE_CLASS'
    conflict_rows=compare_conflicts(data)
    pd.DataFrame(conflict_rows).to_csv(out/'Table1_ExactOverlapAndLabelConflicts.csv',index=False)
    artifact_rows=[]
    for name in CORPORA:artifact_rows.extend(legacy.format_audit(feat[name]['artifact_only'],labels[name],name))
    pd.DataFrame(artifact_rows).to_csv(out/'Table2_SourceArtifactFrequencies.csv',index=False)
    rows=[]; counts=[]; splits=[]; pred_ref={};cal_rows=[]
    for origin in ORIGINS:
        y=labels[origin];g=groups[origin]
        for seed in SEEDS:
            print(f'ORIGIN={origin} SEED={seed}',flush=True)
            split=split_groups(y,g,seed,min_cal_neg=(8 if args.smoke_test else 100))
            # train+cal excludes all held-out test domains but still violates full-corpus independence.
            tc=np.concatenate([split['train'],split['cal']])
            splits += [{'origin':origin,'seed':seed,'partition':part,**label_counts(y[idx],g[idx])} for part,idx in split.items()]
            masks={}
            for dst in CORPORA:
                if dst==origin:continue
                for policy in POLICIES:
                    mask=overlap_mask(origin,dst,policy,tc,data,groups)
                    masks[(dst,policy)]=mask
                    c=label_counts(labels[dst][mask],groups[dst][mask])
                    keep_legit,keep_phish=class_retention(c,labels[dst])
                    quality=policy=='strict_all' and strict_inference_quality(c,labels[dst])
                    counts.append({'origin':origin,'corpus':dst,'seed':seed,'policy':policy,
                                   'excluded_n':int((~mask).sum()),'source_class_retention_legit':keep_legit,
                                   'source_class_retention_phishing':keep_phish,
                                   'primary_interpretability_gate':quality,**c})
            for rep in REPS:
                if rep=='artifact_only':model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,class_weight='balanced',random_state=seed))
                else:model=HistGradientBoostingClassifier(max_leaf_nodes=25,max_depth=5,
                               min_samples_leaf=8 if args.smoke_test else 30,max_iter=18 if args.smoke_test else 130,
                               learning_rate=.07,l2_regularization=1.,early_stopping=False,random_state=seed)
                model.fit(feat[origin][rep][split['train']],y[split['train']])
                p_cal=model.predict_proba(feat[origin][rep][split['cal']])[:,1]
                thresh={tar:cal_threshold(y[split['cal']],p_cal,tar,min_cal_neg=(8 if args.smoke_test else 100)) for tar in CAL_TARGETS}
                for tar,t in thresh.items():
                    calp=p_cal
                    cal_pred=(calp>=t)
                    cal_real=float(np.mean(cal_pred[y[split['cal']]==0]))
                    cal_rows.append({'origin':origin,'seed':seed,'representation':rep,'target_fpr':tar,
                                     'threshold':t,'empirical_calibration_fpr':cal_real,
                                     'n_calibration_negatives':int((y[split['cal']]==0).sum())})
                ix=split['test']
                p_test=model.predict_proba(feat[origin][rep][ix])[:,1]
                for tar,t in thresh.items():
                    r,z=safe_evaluate(y[ix],p_test,t,origin=origin,corpus=origin,seed=seed,
                                       representation=rep,policy='source_internal_group_holdout',target_fpr=tar,
                                       independent_class_diversity=True)
                    rows.append(r)
                for dst in CORPORA:
                    if dst==origin:continue
                    # Generate predictions once for the full corpus and slice for each policy;
                    # this prevents comparing different trained models between overlap rules.
                    probabilities=model.predict_proba(feat[dst][rep])[:,1]
                    for policy in POLICIES:
                        mask=masks[(dst,policy)]
                        c=label_counts(labels[dst][mask],groups[dst][mask])
                        if not (c['n_legit'] and c['n_phishing']):continue
                        for tar,t in thresh.items():
                            r,z=safe_evaluate(labels[dst][mask],probabilities[mask],t,
                                    origin=origin,corpus=dst,seed=seed,representation=rep,policy=policy,
                                    target_fpr=tar,independent_class_diversity=(policy=='strict_all' and strict_inference_quality(c,labels[dst])),
                                    n_legit=c['n_legit'],n_phishing=c['n_phishing'],
                                    domains_legit=c['domains_legit'],domains_phishing=c['domains_phishing'],
                                    source_retention_legit=class_retention(c,labels[dst])[0],
                                    source_retention_phishing=class_retention(c,labels[dst])[1])
                            rows.append(r)
                            if seed==42 and tar==.01 and policy=='strict_all' and rep in ('raw','presentation_ablated'):
                                pred_ref[(origin,dst,rep)]=(labels[dst][mask].copy(),groups[dst][mask].copy(),z.copy())
    df=pd.DataFrame(rows)
    df.to_csv(out/'Table4_AllPerformance.csv',index=False)
    pd.DataFrame(counts).to_csv(out/'Table3_OverlapPolicySampleComposition.csv',index=False)
    pd.DataFrame(splits).to_csv(out/'Table5_TrainCalibrationTestSplits.csv',index=False)
    pd.DataFrame(cal_rows).to_csv(out/'Table6_ThresholdCalibration.csv',index=False)
    summary=summary_rows(rows)
    summary.to_csv(out/'Table7_PrimaryTransferSummary.csv',index=False)
    ci={}
    if not args.smoke_test:
        for origin in ORIGINS:
            for dst in CORPORA:
                if dst==origin:continue
                a=pred_ref.get((origin,dst,'presentation_ablated'))
                b=pred_ref.get((origin,dst,'raw'))
                if a is None or b is None:continue
                ya,ga,za=a;yb,gb,zb=b
                assert np.array_equal(ya,yb) and np.array_equal(ga,gb)
                c=label_counts(ya,ga)
                key=f'{origin}_to_{dst}'
                if strict_inference_quality(c,labels[dst]):ci[key]={'n':int(len(ya)),'domains_by_class':[c['domains_legit'],c['domains_phishing']],
                     'paired_cluster_bootstrap':legacy.cluster_bootstrap(ya,ga,za,zb,reps=400),
                     'note':'Descriptive paired domain-cluster CI for seed 42; no causality or fully independent repeats'}
                else:ci[key]={'not_computed':'Insufficient class/domain diversity or <10% class retention for source-level inference',**c}
    (out/'Table8_PairedClusterBootstrap.json').write_text(json.dumps(ci,indent=2,ensure_ascii=False),encoding='utf8')
    make_plots(df,summary,counts,out)
    manifest={'study':'PhishRoute_v3_decisive','status':'SMOKE_TEST_NO_SCIENCE' if args.smoke_test else 'COMPLETE_REAL_DATA_STUDY',
              'sources':metadata,'seeds':SEEDS,'reps':REPS,'calibration_fpr_targets':CAL_TARGETS,
              'overlap_policies':{ 'strict_all':'Exclude every shared registered domain with the entire training-source corpus; primary inference',
               'training_cal_domains':'Exclude groups in train/cal only; exploratory not source-independent',
               'exact_urls':'Remove exact duplicate URLs only; exploratory','unfiltered':'No filtering; contaminated sensitivity check'},
              'class_minimum_for_inference':MIN_CLASS_FOR_INFERENCE,'class_domain_minimum_for_inference':MIN_GROUPS_FOR_INFERENCE,
              'minimum_class_retention_for_primary_inference':MIN_CLASS_RETENTION,
              'smoke':bool(args.smoke_test),
              'environment':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__},
              'elapsed_seconds':time.perf_counter()-st,
              'limitations':['Source collection bias','Potential contradictory cross-source labels','No trustworthy temporal holdout',
                             'LegitPhish benign class impoverished under strict overlap','Repeat seeds do not constitute three independent studies',
                             'Post-hoc project from earlier v1/v2 results','No claim of causal effect or novel detection model']}
    (out/'MANIFEST.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf8')
    gate={'real_three_source_study':not args.smoke_test,'num_group_splits_per_origin':len(SEEDS),'bidirectional_source_training':True,
          'strict_domain_independent_primary_results':True,'threshold_sensitivity_tested':True,
          'overlap_policy_sensitivity_tested':True,'submission_ready':False,
          'reason':'Academic novelty and validity review required; automatic metrics cannot establish publication fitness.'}
    (out/'EVIDENCE_GATE.json').write_text(json.dumps(gate,indent=2),encoding='utf8')
    generate_brief(summary,rows,metadata,conflict_rows,out)
    (out/'RUN_STATUS.txt').write_text(f"{manifest['status']}\nNo synthetic substitute for missing sources.\nResults are not automatically publication-ready.\n",encoding='utf8')
    shutil.make_archive(str(out),'zip',root_dir=out)
    print('DONE',out.with_suffix('.zip'),'elapsed',manifest['elapsed_seconds'],flush=True)
    if not summary.empty:print(summary[['origin','corpus','representation','fpr_median','recall_median','class_coverage_ok']].to_string(index=False))
    return 0

if __name__=='__main__':
    try:sys.exit(main())
    except Exception:
        import traceback;traceback.print_exc();sys.exit(1)


In [ ]:
import sys,subprocess,pathlib
cmd=[sys.executable,'/kaggle/working/run_decisive_study.py','--out','/kaggle/working/PhishRoute_v3_final']
# If autodetection is ambiguous, specify exact paths here, e.g.:
# cmd += ['--primary','/kaggle/input/your-data/PhishRoute_results.zip',
#         '--external','/kaggle/input/your-data/url_features_extracted1.csv',
#         '--external2','/kaggle/input/your-data/URL-Phish_Dataset.csv']
subprocess.run(cmd,check=True)
print('FINAL ZIP:', pathlib.Path('/kaggle/working/PhishRoute_v3_final.zip'))


In [ ]:
import pandas as pd
from pathlib import Path
out=Path('/kaggle/working/PhishRoute_v3_final')
print((out/'RUN_STATUS.txt').read_text())
print('PRIMARY STRICT-ALL TWO-WAY RESULTS')
display(pd.read_csv(out/'Table7_PrimaryTransferSummary.csv').round(4))
print('Overlap policy sample counts')
display(pd.read_csv(out/'Table3_OverlapPolicySampleComposition.csv').query('seed == 42').head(25))
print('ZIP:',str(out)+'.zip')
